We need to do inference on the test data. Once we have the predictions, we also need to do a posthoc analysis to correctly get the tassel densities for each test image. Another thing to consider here is that we may need to report the mae's seperately for each block, and also entire dataset together.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

Matplotlib is building the font cache; this may take a moment.
2025-07-07 09:17:36.013720: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-07 09:17:37.338125: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-07 09:17:37.338184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-07 09:17:37.502867: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-

In [2]:
# load the trained model
basemodel1_stage2 = tf.keras.models.load_model("models/stage2_basemodel1.keras")

2025-07-07 09:19:20.670687: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31134 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:86:00.0, compute capability: 7.0


In [3]:
# where is the data?

In [4]:
# The input features are at some other location
test_features_location = "seq_2_seq_test_data"

In [5]:
# os.listdir(test_features_location)

In [6]:
# only get the features files
all_test_featurefile_names = [file for file in os.listdir(test_features_location) if file.split(".")[0][-14:] == 'input_features']
all_test_featurefile_names.sort()

In [7]:
all_test_featurefile_names

['block_0103_extracted_input_features.npy',
 'block_0104_extracted_input_features.npy',
 'block_0105_extracted_input_features.npy',
 'block_0106_extracted_input_features.npy',
 'block_0201_extracted_input_features.npy',
 'block_0202_extracted_input_features.npy',
 'block_0205_extracted_input_features.npy',
 'block_0206_extracted_input_features.npy',
 'block_0302_extracted_input_features.npy',
 'block_0303_extracted_input_features.npy',
 'block_0304_extracted_input_features.npy',
 'block_0305_extracted_input_features.npy',
 'block_0306_extracted_input_features.npy']

In [8]:
# We also need the true tassel counts/densities for the test (and other) data. Let's store these in csv files first?
# For the current data, since our windows are non overlapping, we should be able to get the true counts if we sum across the densities of each image?

density_location = "stacked_densities"

In [9]:
all_density_files = os.listdir(density_location)
all_density_files.sort()

In [10]:
all_density_files

['stacked_densities_block_0101.npy',
 'stacked_densities_block_0102.npy',
 'stacked_densities_block_0103.npy',
 'stacked_densities_block_0104.npy',
 'stacked_densities_block_0105.npy',
 'stacked_densities_block_0106.npy',
 'stacked_densities_block_0201.npy',
 'stacked_densities_block_0202.npy',
 'stacked_densities_block_0203.npy',
 'stacked_densities_block_0204.npy',
 'stacked_densities_block_0205.npy',
 'stacked_densities_block_0206.npy',
 'stacked_densities_block_0301.npy',
 'stacked_densities_block_0302.npy',
 'stacked_densities_block_0303.npy',
 'stacked_densities_block_0304.npy',
 'stacked_densities_block_0305.npy',
 'stacked_densities_block_0306.npy']

In [11]:
# do for one npy density file and later do a function for the rest
example_0 = np.load(os.path.join(density_location, all_density_files[0]))

In [12]:
example_0.shape

(910, 7)

In [13]:
true_densities = np.sum(example_0, axis = 0)

In [14]:
pd.DataFrame(["test_im_" + str(i+1) for i in range(len(true_densities))])

,0
0,test_im_1
1,test_im_2
2,test_im_3
3,test_im_4
4,test_im_5
5,test_im_6
6,test_im_7


In [15]:
# Okay, now save these values in a csv file? 
test_df = pd.concat((pd.DataFrame(["test_im_" + str(i) for i in range(len(true_densities))]), pd.DataFrame(true_densities)), axis = 1)

In [16]:
test_df.columns = ["Test_image_name", "True_density"]

In [17]:
test_df

,Test_image_name,True_density
0,test_im_0,59.989377
1,test_im_1,48.977705
2,test_im_2,55.997329
3,test_im_3,44.000000
4,test_im_4,38.000043
5,test_im_5,46.999915
6,test_im_6,32.005661


In [18]:
# save the dataset
test_df.to_csv(os.path.join("all_true_counts", "true_counts_" + str(all_density_files[0].split(".")[0][-4:]) + ".csv"))

In [19]:
all_density_files[0]

'stacked_densities_block_0101.npy'

In [20]:
# Okay, now define a funtion for this? 

In [21]:
def save_true_densities(density_location, file_name, csv_location):
    # load the file
    loaded_file = np.load(os.path.join(density_location, file_name))
    # count the true densities
    true_densities = np.sum(loaded_file, axis = 0)
    # make a dataframe
    test_df = pd.concat((pd.DataFrame(["test_im_" + str(i) for i in range(len(true_densities))]), pd.DataFrame(true_densities)), axis = 1)
    # give column headers for the dataset
    test_df.columns = ["Test_image_name", "True_density"]
    print(test_df)
    # save the dataset
    test_df.to_csv(os.path.join(csv_location, "true_counts_" + str(file_name.split(".")[0][-4:]) + ".csv"), index = False)
    return test_df

In [22]:
%%time
# try this for all blocks
density_loc = "stacked_densities"
csv_loc = "all_true_counts"
all_blocks_true_dfs = []
for file in all_density_files:
    test_df_returned = save_true_densities(density_loc, file, csv_loc)
    all_blocks_true_dfs.append(test_df_returned)

  Test_image_name  True_density
0       test_im_0     59.989377
1       test_im_1     48.977705
2       test_im_2     55.997329
3       test_im_3     44.000000
4       test_im_4     38.000043
5       test_im_5     46.999915
6       test_im_6     32.005661
  Test_image_name  True_density
0       test_im_0     39.995223
1       test_im_1     48.001213
2       test_im_2     51.000046
3       test_im_3     41.005658
4       test_im_4     38.000174
5       test_im_5     39.003503
6       test_im_6     23.000000
  Test_image_name  True_density
0       test_im_0     40.000661
1       test_im_1     39.000001
2       test_im_2     41.000000
3       test_im_3     31.000000
4       test_im_4     32.000000
5       test_im_5     40.002086
6       test_im_6     27.000176
  Test_image_name  True_density
0       test_im_0     33.000000
1       test_im_1     30.000000
2       test_im_2     39.000001
3       test_im_3     40.000000
4       test_im_4     40.998810
5       test_im_5     42.169009
6       

In [23]:
# We have confirmed the true densities in the csv files

In [24]:
# Okay, what next?

In [25]:
# Predict for test data? And also do a posthoc normalization step (considering the generic case to get the tassel densities per test image)

In [26]:
# get the image height and width
image_height = 768
image_width = 1024
print(image_height, image_width)

768 1024


In [27]:
# all_test_featurefile_names

In [28]:
# Do this for a single block of data?
loaded_test_featured = np.load(os.path.join(test_features_location, all_test_featurefile_names[0]))

In [29]:
loaded_test_featured.shape

(910, 13, 32)

In [30]:
pred_vals_test_im_0 = basemodel1_stage2.predict(loaded_test_featured)

29/29 [==============================] - 1s 4ms/step


2025-06-26 12:15:28.461016: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


In [31]:
pred_vals_test_im_0.shape

(910, 7, 1)

In [32]:
# reshape the predicted value, get rid of the final dimension
pred_vals_test_im_0 = pred_vals_test_im_0.reshape(pred_vals_test_im_0.shape[0], pred_vals_test_im_0.shape[1])

In [33]:
pred_vals_test_im_0.shape

(910, 7)

In [34]:
def prediction_on_test_data(pred_values, image_height, image_width, stride = 30, kernel_size = 30):
    # density map
    Density_map = np.zeros((image_height, image_width))

    # counts map
    counts_map = np.zeros((image_height, image_width))
    
    # now, for every window, we will keep adding the values together and also add the counts
    counter = 0
#     need a counter to move into each predicted value in the pred values list
    for ii in range(0, image_height, stride):
        for jj in range(0, image_width, stride):
#         operations for density map
#             get the window of interest
            new_window = Density_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     fill each with the value c_k
            counts_window = np.full((new_window.shape[0], new_window.shape[1]), pred_values[counter])
#     get the shapes of this new window
            cw_height = counts_window.shape[0]
            cw_width = counts_window.shape[1]
#         Do c_k/r_2
            counts_window_new = counts_window/(cw_height*cw_width)
#     This is the value in the window now
            value_window = counts_window_new
#     place the values in the corrsponding area of the density map
            Density_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window + value_window

#         Let's now focus on capturing the counts of the windows
            new_window_c = counts_map[ii:ii + kernel_size,jj:jj+kernel_size]
#     get the counts area
            count = np.ones((new_window_c.shape[0], new_window_c.shape[1]))
#     keep adding the counts to reflect the addition of densities
            counts_map[ii:ii + kernel_size,jj:jj+kernel_size] = new_window_c + count
#     increase the counter
            counter = counter + 1
            
#         get the normalized count
    normalized_counts = np.divide(Density_map, counts_map)
    
#     entire count on the test set
    pred_on_test = np.sum(normalized_counts)
    
#     return the predicted value
    return(pred_on_test, normalized_counts)


In [35]:
def get_final_forecasted_and_true_values(preds_from_model, im_height, im_weight, stride, kernel_size, csv_file_name):
    final_preds_list = []
    for i in range(7):
        preds_per_image = prediction_on_test_data(preds_from_model[:,i], im_height, im_weight, stride , kernel_size)
        final_preds_list.append(preds_per_image[0])
    # make this list  a dataframe
    preds_df = pd.DataFrame(final_preds_list, columns = ['Forecasted_value'])
    
    # Where do we have the true values?
    true_val_location = 'all_true_counts'
    true_value_file = pd.read_csv(os.path.join(true_val_location, csv_file_name))
    
    # compute the mae
    mae_value = mean_absolute_error(true_value_file[['True_density']], preds_df[['Forecasted_value']])
    # compute the rmse
    rmse_value = np.sqrt(mean_squared_error(true_value_file[['True_density']], preds_df[['Forecasted_value']]))
    # pearsonr
    pearson_value = pearsonr(np.array(true_value_file[['True_density']]).reshape(-1), np.array(preds_df[['Forecasted_value']]).reshape(-1))
    # r2score
    r2score_value = r2_score(true_value_file[['True_density']], preds_df[['Forecasted_value']])
    # attach the true and the forecasted values together
    final_df = pd.concat((true_value_file, preds_df), axis = 1)
    # final df location
    final_loc = 'all_predicted_counts'
    # save this file
    final_df.to_csv(os.path.join(final_loc, csv_file_name.split(".")[0][-4:] + '.csv'), index = False)
    all_metrics = [mae_value, rmse_value, pearson_value, r2score_value]

    return(final_preds_list, all_metrics, final_df)

In [36]:
all_csv_files = os.listdir("all_true_counts")
all_csv_files.sort()

In [37]:
all_test_blocks = [file.split(".")[0].split("_")[1] for file in all_test_featurefile_names]
all_test_blocks.sort()

In [38]:
all_test_csv_files = [file for file in all_csv_files if file.split(".")[0].split("_")[-1] in all_test_blocks]
all_test_csv_files.sort()

In [39]:
all_test_csv_files

['true_counts_0103.csv',
 'true_counts_0104.csv',
 'true_counts_0105.csv',
 'true_counts_0106.csv',
 'true_counts_0201.csv',
 'true_counts_0202.csv',
 'true_counts_0205.csv',
 'true_counts_0206.csv',
 'true_counts_0302.csv',
 'true_counts_0303.csv',
 'true_counts_0304.csv',
 'true_counts_0305.csv',
 'true_counts_0306.csv']

In [40]:
preds_block_0103, metrics_0103, preds_df_0103 = get_final_forecasted_and_true_values(pred_vals_test_im_0, image_height, image_width, 30, 30, all_test_csv_files[0])

In [41]:
preds_block_0103

[50.22763431226667,
 48.5403152419108,
 45.61077332272724,
 40.097399247829216,
 33.05866200638142,
 25.429728905873397,
 17.76128263797117]

In [42]:
metrics_0103

[8.335053235980084,
 9.24632730769745,
 PearsonRResult(statistic=0.624833211862317, pvalue=0.13355038228313598),
 -2.1926645324572203]

In [43]:
preds_df_0103

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.000661,50.227634
1,test_im_1,39.000001,48.540315
2,test_im_2,41.000000,45.610773
3,test_im_3,31.000000,40.097399
4,test_im_4,32.000000,33.058662
5,test_im_5,40.002086,25.429729
6,test_im_6,27.000176,17.761283


In [44]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_im_0, axis = 0)

array([50.22763 , 48.540325, 45.61075 , 40.097424, 33.058666, 25.429712,
       17.761292], dtype=float32)

In [45]:
# note the values match

In [46]:
# Now do this for all the test blocks

Block 0104

In [47]:
all_test_featurefile_names[1]

'block_0104_extracted_input_features.npy'

In [48]:
# Do this for a single block of data?
loaded_test_features_0104 = np.load(os.path.join(test_features_location, all_test_featurefile_names[1]))

In [49]:
loaded_test_features_0104.shape

(910, 13, 32)

In [50]:
pred_vals_test_0104 = basemodel1_stage2.predict(loaded_test_features_0104)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0104 = pred_vals_test_0104.reshape(pred_vals_test_0104.shape[0], pred_vals_test_0104.shape[1])

29/29 [==============================] - 0s 3ms/step


In [51]:
pred_vals_test_0104.shape

(910, 7)

In [52]:
all_test_csv_files[1]

'true_counts_0104.csv'

In [53]:
preds_block_0104, metrics_0104, preds_df_0104 = get_final_forecasted_and_true_values(pred_vals_test_0104, image_height, image_width, 30, 30, all_test_csv_files[1])

In [54]:
preds_block_0104

[32.08162200726474,
 27.252287351258225,
 25.329792245401297,
 22.74631101306789,
 19.493989620710323,
 15.839668546035803,
 11.912304465187967]

In [55]:
metrics_0104

[14.359594561941192,
 16.80138731021287,
 PearsonRResult(statistic=-0.1785489919169737, pvalue=0.7016942251011925),
 -10.794052223515202]

In [56]:
preds_df_0104

,Test_image_name,True_density,Forecasted_value
0,test_im_0,33.000000,32.081622
1,test_im_1,30.000000,27.252287
2,test_im_2,39.000001,25.329792
3,test_im_3,40.000000,22.746311
4,test_im_4,40.998810,19.493990
5,test_im_5,42.169009,15.839669
6,test_im_6,30.005317,11.912304


In [57]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0104, axis = 0)

array([32.08164 , 27.252274, 25.329803, 22.74631 , 19.493982, 15.83967 ,
       11.912293], dtype=float32)

Block 0105

In [58]:
all_test_featurefile_names[2]

'block_0105_extracted_input_features.npy'

In [59]:
# Do this for a single block of data?
loaded_test_features_0105 = np.load(os.path.join(test_features_location, all_test_featurefile_names[2]))

In [60]:
loaded_test_features_0105.shape

(910, 13, 32)

In [61]:
pred_vals_test_0105 = basemodel1_stage2.predict(loaded_test_features_0105)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0105 = pred_vals_test_0105.reshape(pred_vals_test_0105.shape[0], pred_vals_test_0105.shape[1])

29/29 [==============================] - 0s 4ms/step


In [62]:
pred_vals_test_0105.shape

(910, 7)

In [63]:
all_test_csv_files[2]

'true_counts_0105.csv'

In [64]:
preds_block_0105, metrics_0105, preds_df_0105 = get_final_forecasted_and_true_values(pred_vals_test_0105, image_height, image_width, 30, 30, all_test_csv_files[2])

In [65]:
preds_block_0105

[38.124812650858985,
 35.28080016309289,
 33.40844965416321,
 29.919452072015446,
 25.28756180032149,
 20.090034827886576,
 14.660760168015639]

In [66]:
metrics_0105

[12.604687576117291,
 14.220824486142778,
 PearsonRResult(statistic=0.7400135610923619, pvalue=0.05721352222462014),
 -1.1336263823635928]

In [67]:
preds_df_0105

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.000001,38.124813
1,test_im_1,46.002743,35.280800
2,test_im_2,58.000696,33.408450
3,test_im_3,41.000032,29.919452
4,test_im_4,41.001190,25.287562
5,test_im_5,36.000022,20.090035
6,test_im_6,23.000000,14.660760


In [68]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0105, axis = 0)

array([38.124783, 35.2808  , 33.408443, 29.919453, 25.287554, 20.090034,
       14.660765], dtype=float32)

Block 0106

In [69]:
all_test_featurefile_names[3]

'block_0106_extracted_input_features.npy'

In [70]:
# Do this for a single block of data?
loaded_test_features_0106 = np.load(os.path.join(test_features_location, all_test_featurefile_names[3]))

In [71]:
loaded_test_features_0106.shape

(910, 13, 32)

In [72]:
pred_vals_test_0106 = basemodel1_stage2.predict(loaded_test_features_0106)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0106 = pred_vals_test_0106.reshape(pred_vals_test_0106.shape[0], pred_vals_test_0106.shape[1])

29/29 [==============================] - 0s 3ms/step


In [73]:
pred_vals_test_0106.shape

(910, 7)

In [74]:
all_test_csv_files[3]

'true_counts_0106.csv'

In [75]:
preds_block_0106, metrics_0106, preds_df_0106 = get_final_forecasted_and_true_values(pred_vals_test_0106, image_height, image_width, 30, 30, all_test_csv_files[3])

In [76]:
preds_block_0106

[38.7761359965898,
 35.40147182810406,
 33.560026579839956,
 30.399404405962365,
 26.303851584394813,
 21.753950413751227,
 16.990508539591943]

In [77]:
metrics_0106

[12.685384588048148,
 15.327955800470404,
 PearsonRResult(statistic=-0.20522090140598429, pvalue=0.6588960059771207),
 -18.712249523248378]

In [78]:
preds_df_0106

,Test_image_name,True_density,Forecasted_value
0,test_im_0,38.999667,38.776136
1,test_im_1,38.999989,35.401472
2,test_im_2,45.000000,33.560027
3,test_im_3,39.986887,30.399404
4,test_im_4,42.999997,26.303852
5,test_im_5,47.996502,21.753950
6,test_im_6,38.000000,16.990509


In [79]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0106, axis = 0)

array([38.77614 , 35.40148 , 33.560036, 30.399408, 26.30384 , 21.753948,
       16.990505], dtype=float32)

Block 0201

In [80]:
all_test_featurefile_names[4]

'block_0201_extracted_input_features.npy'

In [81]:
# Do this for a single block of data?
loaded_test_features_0201 = np.load(os.path.join(test_features_location, all_test_featurefile_names[4]))

In [82]:
loaded_test_features_0201.shape

(910, 13, 32)

In [83]:
pred_vals_test_0201 = basemodel1_stage2.predict(loaded_test_features_0201)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0201 = pred_vals_test_0201.reshape(pred_vals_test_0201.shape[0], pred_vals_test_0201.shape[1])

29/29 [==============================] - 0s 4ms/step


In [84]:
pred_vals_test_0201.shape

(910, 7)

In [85]:
all_test_csv_files[4]

'true_counts_0201.csv'

In [86]:
preds_block_0201, metrics_0201, preds_df_0201 = get_final_forecasted_and_true_values(pred_vals_test_0201, image_height, image_width, 30, 30, all_test_csv_files[4])

In [87]:
preds_block_0201

[46.65342369555174,
 44.53097120608166,
 42.15500935794515,
 37.67858126117752,
 31.81180195951557,
 25.30309498534046,
 18.614319198078388]

In [88]:
metrics_0201

[5.365644270484684,
 6.889152133011415,
 PearsonRResult(statistic=0.9143520918031363, pvalue=0.003935703412038591),
 -0.31386208952916106]

In [89]:
preds_df_0201

,Test_image_name,True_density,Forecasted_value
0,test_im_0,45.000217,46.653424
1,test_im_1,45.000040,44.530971
2,test_im_2,47.000001,42.155009
3,test_im_3,38.000000,37.678581
4,test_im_4,42.000041,31.811802
5,test_im_5,35.000000,25.303095
6,test_im_6,29.000000,18.614319


In [90]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0201, axis = 0)

array([46.653435, 44.530975, 42.15502 , 37.678593, 31.81179 , 25.303078,
       18.614304], dtype=float32)

Block 0202

In [91]:
all_test_featurefile_names[5]

'block_0202_extracted_input_features.npy'

In [92]:
# Do this for a single block of data?
loaded_test_features_0202 = np.load(os.path.join(test_features_location, all_test_featurefile_names[5]))

In [93]:
loaded_test_features_0202.shape

(910, 13, 32)

In [94]:
pred_vals_test_0202 = basemodel1_stage2.predict(loaded_test_features_0202)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0202 = pred_vals_test_0202.reshape(pred_vals_test_0202.shape[0], pred_vals_test_0202.shape[1])

29/29 [==============================] - 0s 4ms/step


In [95]:
pred_vals_test_0202.shape

(910, 7)

In [96]:
all_test_csv_files[5]

'true_counts_0202.csv'

In [97]:
preds_block_0202, metrics_0202, preds_df_0202 = get_final_forecasted_and_true_values(pred_vals_test_0202, image_height, image_width, 30, 30, all_test_csv_files[5])

In [98]:
preds_block_0202

[37.49252903273373,
 34.83790388934338,
 32.677635598453975,
 28.662412083559435,
 23.346798654093938,
 17.397186188499294,
 11.23110518939815]

In [99]:
metrics_0202

[8.627006557907857,
 10.525520534931363,
 PearsonRResult(statistic=0.45439476571981197, pvalue=0.3057021107354969),
 -31.312591407703245]

In [100]:
preds_df_0202

,Test_image_name,True_density,Forecasted_value
0,test_im_0,17.999960,37.492529
1,test_im_1,21.000000,34.837904
2,test_im_2,23.000000,32.677636
3,test_im_3,20.999982,28.662412
4,test_im_4,21.000000,23.346799
5,test_im_5,18.000000,17.397186
6,test_im_6,18.000000,11.231105


In [101]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0202, axis = 0)

array([37.492504, 34.837887, 32.677616, 28.662416, 23.346813, 17.397182,
       11.231104], dtype=float32)

Block 0205

In [102]:
all_test_featurefile_names[6]

'block_0205_extracted_input_features.npy'

In [103]:
# Do this for a single block of data?
loaded_test_features_0205 = np.load(os.path.join(test_features_location, all_test_featurefile_names[6]))

In [104]:
loaded_test_features_0205.shape

(910, 13, 32)

In [105]:
pred_vals_test_0205 = basemodel1_stage2.predict(loaded_test_features_0205)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0205 = pred_vals_test_0205.reshape(pred_vals_test_0205.shape[0], pred_vals_test_0205.shape[1])

29/29 [==============================] - 0s 4ms/step


In [106]:
pred_vals_test_0205.shape

(910, 7)

In [107]:
all_test_csv_files[6]

'true_counts_0205.csv'

In [108]:
preds_block_0205, metrics_0205, preds_df_0205 = get_final_forecasted_and_true_values(pred_vals_test_0205, image_height, image_width, 30, 30, all_test_csv_files[6])

In [109]:
preds_block_0205

[44.771807867104144,
 40.988737975579625,
 38.26220791077132,
 34.05332213808572,
 28.939854421141447,
 23.495907993916504,
 17.992123746042264]

In [110]:
metrics_0205

[8.434175117997546,
 10.218100013518463,
 PearsonRResult(statistic=0.8235192578078572, pvalue=0.02279564201253769),
 -5.362933042345562]

In [111]:
preds_df_0205

,Test_image_name,True_density,Forecasted_value
0,test_im_0,44.000000,44.771808
1,test_im_1,42.000001,40.988738
2,test_im_2,45.000000,38.262208
3,test_im_3,43.000000,34.053322
4,test_im_4,39.000000,28.939854
5,test_im_5,40.999915,23.495908
6,test_im_6,31.999656,17.992124


In [112]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0205, axis = 0)

array([44.771767, 40.988724, 38.262215, 34.053295, 28.939842, 23.495897,
       17.992125], dtype=float32)

Block 0206

In [113]:
all_test_featurefile_names[7]

'block_0206_extracted_input_features.npy'

In [114]:
# Do this for a single block of data?
loaded_test_features_0206 = np.load(os.path.join(test_features_location, all_test_featurefile_names[7]))

In [115]:
loaded_test_features_0206.shape

(910, 13, 32)

In [116]:
pred_vals_test_0206 = basemodel1_stage2.predict(loaded_test_features_0206)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0206 = pred_vals_test_0206.reshape(pred_vals_test_0206.shape[0], pred_vals_test_0206.shape[1])

29/29 [==============================] - 0s 4ms/step


In [117]:
pred_vals_test_0206.shape

(910, 7)

In [118]:
all_test_csv_files[7]

'true_counts_0206.csv'

In [119]:
preds_block_0206, metrics_0206, preds_df_0206 = get_final_forecasted_and_true_values(pred_vals_test_0206, image_height, image_width, 30, 30, all_test_csv_files[7])

In [120]:
preds_block_0206

[42.07604214502135,
 39.00370849072741,
 36.72401064335034,
 32.77249749679905,
 27.70846712369013,
 22.145127929168467,
 16.422374631096446]

In [121]:
metrics_0206

[1.7515012449064247,
 1.944210955285652,
 PearsonRResult(statistic=0.977129452637894, pvalue=0.00015007327423711674),
 0.9522113780136257]

In [122]:
preds_df_0206

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.999838,42.076042
1,test_im_1,41.998810,39.003708
2,test_im_2,39.000067,36.724011
3,test_im_3,32.000003,32.772497
4,test_im_4,25.000352,27.708467
5,test_im_5,23.000040,22.145128
6,test_im_6,18.000000,16.422375


In [123]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0206, axis = 0)

array([42.076065, 39.0037  , 36.723988, 32.77249 , 27.708483, 22.14513 ,
       16.422377], dtype=float32)

Block 0302

In [124]:
all_test_featurefile_names[8]

'block_0302_extracted_input_features.npy'

In [125]:
# Do this for a single block of data?
loaded_test_features_0302 = np.load(os.path.join(test_features_location, all_test_featurefile_names[8]))

In [126]:
loaded_test_features_0302.shape

(910, 13, 32)

In [127]:
pred_vals_test_0302 = basemodel1_stage2.predict(loaded_test_features_0302)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0302 = pred_vals_test_0302.reshape(pred_vals_test_0302.shape[0], pred_vals_test_0302.shape[1])

29/29 [==============================] - 0s 4ms/step


In [128]:
pred_vals_test_0302.shape

(910, 7)

In [129]:
all_test_csv_files[8]

'true_counts_0302.csv'

In [130]:
preds_block_0302, metrics_0302, preds_df_0302 = get_final_forecasted_and_true_values(pred_vals_test_0302, image_height, image_width, 30, 30, all_test_csv_files[8])

In [131]:
preds_block_0302

[48.23320928294743,
 46.70538552566914,
 44.170999177287946,
 39.117594588541266,
 32.528739852855914,
 25.30191781312839,
 17.97081686285084]

In [132]:
metrics_0302

[10.85912019920521,
 13.140941939193315,
 PearsonRResult(statistic=0.7767284858326768, pvalue=0.03994950680158965),
 -5.601959329459416]

In [133]:
preds_df_0302

,Test_image_name,True_density,Forecasted_value
0,test_im_0,49.000000,48.233209
1,test_im_1,49.000005,46.705386
2,test_im_2,53.999570,44.170999
3,test_im_3,50.042349,39.117595
4,test_im_4,43.000009,32.528740
5,test_im_5,48.000572,25.301918
6,test_im_6,36.999999,17.970817


In [134]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0302, axis = 0)

array([48.23321 , 46.705364, 44.171013, 39.117588, 32.528736, 25.301916,
       17.97081 ], dtype=float32)

Block 0303

In [135]:
all_test_featurefile_names[9]

'block_0303_extracted_input_features.npy'

In [136]:
# Do this for a single block of data?
loaded_test_features_0303 = np.load(os.path.join(test_features_location, all_test_featurefile_names[9]))

In [137]:
loaded_test_features_0303.shape

(910, 13, 32)

In [138]:
pred_vals_test_0303 = basemodel1_stage2.predict(loaded_test_features_0303)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0303 = pred_vals_test_0303.reshape(pred_vals_test_0303.shape[0], pred_vals_test_0303.shape[1])

29/29 [==============================] - 0s 4ms/step


In [139]:
pred_vals_test_0303.shape

(910, 7)

In [140]:
all_test_csv_files[9]

'true_counts_0303.csv'

In [141]:
preds_block_0303, metrics_0303, preds_df_0303 = get_final_forecasted_and_true_values(pred_vals_test_0303, image_height, image_width, 30, 30, all_test_csv_files[9])

In [142]:
preds_block_0303

[44.7654627848533,
 43.06852719111822,
 40.70768201789974,
 35.98287162413496,
 29.7530293179585,
 22.85709774226371,
 15.80860251137895]

In [143]:
metrics_0303

[5.583252037908067,
 6.597453796847326,
 PearsonRResult(statistic=0.953116459062906, pvalue=0.0008912691513063459),
 0.3055035531877853]

In [144]:
preds_df_0303

,Test_image_name,True_density,Forecasted_value
0,test_im_0,49.000171,44.765463
1,test_im_1,46.000007,43.068527
2,test_im_2,42.999982,40.707682
3,test_im_3,38.999993,35.982872
4,test_im_4,36.025884,29.753029
5,test_im_5,36.000000,22.857098
6,test_im_6,23.000000,15.808603


In [145]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0303, axis = 0)

array([44.765453, 43.068512, 40.70771 , 35.98282 , 29.75305 , 22.857098,
       15.808605], dtype=float32)

Block 0304

In [146]:
all_test_featurefile_names[10]

'block_0304_extracted_input_features.npy'

In [147]:
# Do this for a single block of data?
loaded_test_features_0304 = np.load(os.path.join(test_features_location, all_test_featurefile_names[10]))

In [148]:
loaded_test_features_0304.shape

(910, 13, 32)

In [149]:
pred_vals_test_0304 = basemodel1_stage2.predict(loaded_test_features_0304)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0304 = pred_vals_test_0304.reshape(pred_vals_test_0304.shape[0], pred_vals_test_0304.shape[1])

29/29 [==============================] - 0s 3ms/step


In [150]:
pred_vals_test_0304.shape

(910, 7)

In [151]:
all_test_csv_files[10]

'true_counts_0304.csv'

In [152]:
preds_block_0304, metrics_0304, preds_df_0304 = get_final_forecasted_and_true_values(pred_vals_test_0304, image_height, image_width, 30, 30, all_test_csv_files[10])

In [153]:
preds_block_0304

[38.480972118699356,
 35.324970193792836,
 33.1934141427069,
 29.480667361710267,
 24.635525455980705,
 19.234443159056234,
 13.618538790442631]

In [154]:
metrics_0304

[9.713631841768931,
 10.635048644028943,
 PearsonRResult(statistic=0.786612168486665, pvalue=0.035878652150256235),
 -2.0925014692956685]

In [155]:
preds_df_0304

,Test_image_name,True_density,Forecasted_value
0,test_im_0,37.000000,38.480972
1,test_im_1,41.002057,35.324970
2,test_im_2,42.999998,33.193414
3,test_im_3,41.999955,29.480667
4,test_im_4,38.000000,24.635525
5,test_im_5,34.000000,19.234443
6,test_im_6,24.000000,13.618539


In [156]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0304, axis = 0)

array([38.480965, 35.32496 , 33.1934  , 29.480669, 24.635508, 19.234447,
       13.618547], dtype=float32)

Block 0305

In [157]:
all_test_featurefile_names[11]

'block_0305_extracted_input_features.npy'

In [158]:
# Do this for a single block of data?
loaded_test_features_0305 = np.load(os.path.join(test_features_location, all_test_featurefile_names[11]))

In [159]:
loaded_test_features_0305.shape

(910, 13, 32)

In [160]:
pred_vals_test_0305 = basemodel1_stage2.predict(loaded_test_features_0305)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0305 = pred_vals_test_0305.reshape(pred_vals_test_0305.shape[0], pred_vals_test_0305.shape[1])

29/29 [==============================] - 0s 3ms/step


In [161]:
pred_vals_test_0305.shape

(910, 7)

In [162]:
all_test_csv_files[11]

'true_counts_0305.csv'

In [163]:
preds_block_0305, metrics_0305, preds_df_0305 = get_final_forecasted_and_true_values(pred_vals_test_0305, image_height, image_width, 30, 30, all_test_csv_files[11])

In [164]:
preds_block_0305

[43.69099521886983,
 40.88632462589787,
 38.47640359730118,
 34.18154177534073,
 28.658480212201557,
 22.595982422354005,
 16.389795243596993]

In [165]:
metrics_0305

[4.315652328635876,
 4.689120524016306,
 PearsonRResult(statistic=0.8736774750217279, pvalue=0.010165168296327971),
 0.6021399675072722]

In [166]:
preds_df_0305

,Test_image_name,True_density,Forecasted_value
0,test_im_0,46.000000,43.690995
1,test_im_1,35.000000,40.886325
2,test_im_2,37.000000,38.476404
3,test_im_3,30.000652,34.181542
4,test_im_4,35.001190,28.658480
5,test_im_5,28.999994,22.595982
6,test_im_6,20.000018,16.389795


In [167]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0305, axis = 0)

array([43.690983, 40.886303, 38.476395, 34.181545, 28.658451, 22.595978,
       16.389778], dtype=float32)

Block 0306

In [168]:
all_test_featurefile_names[12]

'block_0306_extracted_input_features.npy'

In [169]:
# Do this for a single block of data?
loaded_test_features_0306 = np.load(os.path.join(test_features_location, all_test_featurefile_names[12]))

In [170]:
loaded_test_features_0306.shape

(910, 13, 32)

In [171]:
pred_vals_test_0306 = basemodel1_stage2.predict(loaded_test_features_0306)
# reshape the predicted value, get rid of the final dimension
pred_vals_test_0306 = pred_vals_test_0306.reshape(pred_vals_test_0306.shape[0], pred_vals_test_0306.shape[1])

29/29 [==============================] - 0s 3ms/step


In [172]:
pred_vals_test_0306.shape

(910, 7)

In [173]:
all_test_csv_files[12]

'true_counts_0306.csv'

In [174]:
preds_block_0306, metrics_0306, preds_df_0306 = get_final_forecasted_and_true_values(pred_vals_test_0306, image_height, image_width, 30, 30, all_test_csv_files[12])

In [175]:
preds_block_0306

[45.37004971306669,
 42.748658889725476,
 40.234908522756214,
 35.70727932534207,
 29.95809820112081,
 23.72679558892976,
 17.41257758232007]

In [176]:
metrics_0306

[4.726210705895471,
 5.799291260308265,
 PearsonRResult(statistic=0.8547933299277203, pvalue=0.014248610261599704),
 0.4901916986734971]

In [177]:
preds_df_0306

,Test_image_name,True_density,Forecasted_value
0,test_im_0,41.000009,45.370050
1,test_im_1,41.000003,42.748659
2,test_im_2,43.006851,40.234909
3,test_im_3,39.997604,35.707279
4,test_im_4,40.000000,29.958098
5,test_im_5,32.999982,23.726796
6,test_im_6,18.000000,17.412578


In [178]:
# alt method of getting preds - windows are not overlapping
np.sum(pred_vals_test_0306, axis = 0)

array([45.370037, 42.74866 , 40.23489 , 35.707275, 29.958082, 23.7268  ,
       17.41259 ], dtype=float32)

In [179]:
# Get the average metrics also on the entire test data? For future use maybe

In [180]:
# Since we have the preds and true values all stored here  - "all_predicted_counts", let's use that to get the final metrics on the entire test space.

In [181]:
test_true_and_preds_path = "all_predicted_counts"

In [182]:
all_contents_here = os.listdir(test_true_and_preds_path)
all_contents_here.sort()

In [183]:
all_contents_here

['.ipynb_checkpoints',
 '0103.csv',
 '0104.csv',
 '0105.csv',
 '0106.csv',
 '0201.csv',
 '0202.csv',
 '0205.csv',
 '0206.csv',
 '0302.csv',
 '0303.csv',
 '0304.csv',
 '0305.csv',
 '0306.csv']

In [184]:
all_csv_files = [file for file in all_contents_here if file.split(".")[-1] == "csv"]

In [185]:
# all_csv_files

In [186]:
# load all csv files?
csv_files_all = []
for file in all_csv_files:
    loaded_file = pd.read_csv(os.path.join(test_true_and_preds_path, file))
    csv_files_all.append(loaded_file)

In [187]:
all_test_preds_and_true_densities = pd.concat(csv_files_all)

In [188]:
all_test_preds_and_true_densities.head()

,Test_image_name,True_density,Forecasted_value
0,test_im_0,40.000661,50.227634
1,test_im_1,39.000001,48.540315
2,test_im_2,41.000000,45.610773
3,test_im_3,31.000000,40.097399
4,test_im_4,32.000000,33.058662


In [189]:
all_test_preds_and_true_densities.shape

(91, 3)

In [190]:
# mae
all_test_mae = mean_absolute_error(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
all_test_mae

8.258531866676675

In [191]:
# mse
all_test_mse = mean_squared_error(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
np.sqrt(all_test_mse)

10.584206041245345

In [192]:
# r2
all_test_r2_score = r2_score(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
all_test_r2_score

-0.3959369213944435

In [193]:
# pearsonr
all_test_pearson = pearsonr(all_test_preds_and_true_densities['True_density'], all_test_preds_and_true_densities['Forecasted_value'])
all_test_pearson[0]

0.566185669303316